# Vigade analüüsi notebook

Selles märkmikus analüüsime `tagasiside_log.csv` logis leiduvaid halbu päringuid. Eesmärk on välja tuua juhtumid, millele süsteem vastas halvasti, ning arvutada iga vahesammu (metaandmete filtreerimine, RAGi vektorotsing, LLMi vastuse genereerimine) poolt tehtud vigade koguarv ja osakaal kõikidest vigadest.

In [11]:
import pandas as pd

# 1. Loeme andmed
df = pd.read_csv('tagasiside_log.csv')

# 2. Eraldame negatiivse hinnanguga ('👎 Halb') päringud
df_bad = df[df['Hinnang'] == '👎 Halb'].copy()
print(f"Kokku leiti {len(df_bad)} vigast päringut.")

# Kuvame halvad juhtumid (kasutaja päring ja milles seisnes viga)
display(df_bad[['Aeg', 'Kasutaja päring', 'Veatüüp']])

Kokku leiti 8 vigast päringut.


,Aeg,Kasutaja päring,Veatüüp
2,2026-03-01 19:23:29,Tahan õppida närvivõrkude kohta,Otsing leidis valed ained (RAG viga)
3,2026-03-01 19:24:15,Tahan teha origamit,LLM hallutsineeris/vastas valesti
4,2026-03-01 19:27:24,Tahan ehitada lumememme,Otsing leidis valed ained (RAG viga)
5,2026-03-01 23:53:09,Soovin õppida Pythonis programmeerimist,Otsing leidis valed ained (RAG viga)
6,2026-03-02 00:03:24,Tahan õppida uisutamist,Otsing leidis valed ained (RAG viga)
7,2026-03-02 00:09:47,soovin ehitada iglu,Otsing leidis valed ained (RAG viga)
8,2026-03-02 00:10:26,võrkpalli mängimine,Otsing leidis valed ained (RAG viga)
9,2026-03-02 00:11:43,"Oman saksa keeles B2 taset, mis kursuseid võik...",Otsing leidis valed ained (RAG viga)


## Vahesammude vigade koondtabel

Rakenduse vigade tüübid jagunevad kolmeks põhisammuks:
1. **Metaandmete filtreerimine** $\rightarrow$ *Filtrid olid liiga karmid/valed*
2. **RAGi vektorotsing** $\rightarrow$ *Otsing leidis valed ained (RAG viga)*
3. **LLMi vastuse genereerimine** $\rightarrow$ *LLM hallutsineeris/vastas valesti*

In [12]:
# Seostame logitud veatüübid süsteemi vahesammudega
error_mapping = {
    'Filtrid olid liiga karmid/valed': 'Metaandmete filtreerimine',
    'Otsing leidis valed ained (RAG viga)': 'RAGi vektorotsing',
    'LLM hallutsineeris/vastas valesti': 'LLMi vastuse genereerimine'
}

df_bad['Vahesamm'] = df_bad['Veatüüp'].map(error_mapping).fillna('Määramata/Teadmata')

# Arvutame vigade koguarvud vastavalt vahesammule
error_counts = df_bad['Vahesamm'].value_counts().reset_index()
error_counts.columns = ['Vahesamm', 'Vigade koguarv']

# Lisame analüüsi kindlasti kõik kolm sammu isegi siis, kui nende väärtus on hetkel 0
all_steps = pd.DataFrame({
    'Vahesamm': ['Metaandmete filtreerimine', 'RAGi vektorotsing', 'LLMi vastuse genereerimine']
})

summary = pd.merge(all_steps, error_counts, on='Vahesamm', how='left').fillna(0)
summary['Vigade koguarv'] = summary['Vigade koguarv'].astype(int)

# Arvutame vigade osakaalu (protsendi)
total_errors = summary['Vigade koguarv'].sum()

if total_errors > 0:
    summary['Protsent (%)'] = (summary['Vigade koguarv'] / total_errors * 100).round(1)
else:
    summary['Protsent (%)'] = 0.0

display(summary)

,Vahesamm,Vigade koguarv,Protsent (%)
0,Metaandmete filtreerimine,0,0.0
1,RAGi vektorotsing,7,87.5
2,LLMi vastuse genereerimine,1,12.5


# Vigade analüüs

Järgnev analüüs võtab kokku failis `tagasiside_log.csv` olevad halvad juhtumid ja arvutab iga vahesammu (metaandmete filtreerimine, RAGi vektorotsing, LLMi vastuse genereerimine) vigade koguarvu ja protsendi.

In [13]:
import pandas as pd

# Loeme logid sisse
df = pd.read_csv('tagasiside_log.csv')

# Filtreerime välja halva hinnanguga juhtumid
bad_cases = df[df['Hinnang'] == '👎 Halb'].copy()
bad_cases.reset_index(drop=True, inplace=True)

print(f"Leiti {len(bad_cases)} halba juhtumit.")

Leiti 8 halba juhtumit.


## Halvad juhtumid (tabelina)

In [14]:
# Kuvame halvad juhtumid. Lisame tabelisse olulised veerud.
display_cols = ['Kasutaja päring', 'Filtrid', 'Leitud ID-d', 'Veatüüp']
bad_cases[display_cols].style.set_properties(**{'text-align': 'left'})

,Kasutaja päring,Filtrid,Leitud ID-d,Veatüüp
0,Tahan õppida närvivõrkude kohta,filtrid puuduvad,"['ARFS.01.069', 'MVBS.03.006', 'ARNR.01.024', 'ARNR.01.028', 'SVHI.03.035']",Otsing leidis valed ained (RAG viga)
1,Tahan teha origamit,filtrid puuduvad,"['LTMS.00.086', 'LOKT.09.015', 'LTKT.09.003', 'MVSF.00.007', 'LOKT.09.042']",LLM hallutsineeris/vastas valesti
2,Tahan ehitada lumememme,filtrid puuduvad,"['P2VK.04.292', 'LOTI.05.023', 'HVKU.04.053', 'LTFY.02.010', 'HVVK.04.098']",Otsing leidis valed ained (RAG viga)
3,Soovin õppida Pythonis programmeerimist,filtrid puuduvad,"['HVEE.04.028', 'P2NC.01.083_1', 'P2NC.01.083', 'LTOM.02.041', 'SVNC.00.228']",Otsing leidis valed ained (RAG viga)
4,Tahan õppida uisutamist,filtrid puuduvad,"['KKSP.05.261', 'KKSP.04.102_1', 'KKSP.05.265', 'KKSP.04.102', 'LOFY.05.004']",Otsing leidis valed ained (RAG viga)
5,soovin ehitada iglu,filtrid puuduvad,"['HVVK.04.110', 'HVVK.04.098', 'HVVK.04.098_1', 'HVAJ.01.004', 'LOTI.05.021']",Otsing leidis valed ained (RAG viga)
6,võrkpalli mängimine,filtrid puuduvad,"['KKSP.05.287', 'KKSP.05.309', 'ARSM.01.045', 'LOTI.05.023', 'MVSF.00.005']",Otsing leidis valed ained (RAG viga)
7,"Oman saksa keeles B2 taset, mis kursuseid võiksin edasi võtta?",filtrid puuduvad,"['FLKE.02.278', 'HVLC.04.013', 'HVLC.04.011', 'HVLC.04.012', 'HVLC.04.010']",Otsing leidis valed ained (RAG viga)


## Vigade koondarvud ja protsendid

Veatüübid ja nende vastavus vahesammudele:
- **Filtrid olid liiga karmid/valed** -> Metaandmete filtreerimine
- **Otsing leidis valed ained (RAG viga)** -> RAGi vektorotsing
- **LLM hallutsineeris/vastas valesti** -> LLMi vastuse genereerimine

In [15]:
total_bad = len(bad_cases)

error_mapping = {
    'Filtrid olid liiga karmid/valed': 'Metaandmete filtreerimine',
    'Otsing leidis valed ained (RAG viga)': 'RAGi vektorotsing',
    'LLM hallutsineeris/vastas valesti': 'LLMi vastuse genereerimine'
}

# Asendame logi tekstid kokkulepitud kategooriatega
bad_cases['Vahesamm'] = bad_cases['Veatüüp'].map(error_mapping).fillna('Teadmata/Määramata viga')

# Vastava vahesammu sageduse loendus
error_counts = bad_cases['Vahesamm'].value_counts().reset_index()
error_counts.columns = ['Vahesamm', 'Vigade koguarv']

# Lisame kõik oodatud kategooriad juhuks, kui mõnda viga pole kordagi esinenud
all_steps = pd.DataFrame({'Vahesamm': ['Metaandmete filtreerimine', 'RAGi vektorotsing', 'LLMi vastuse genereerimine']})
error_summary = pd.merge(all_steps, error_counts, on='Vahesamm', how='left').fillna(0)
error_summary['Vigade koguarv'] = error_summary['Vigade koguarv'].astype(int)

# Protsendi arvutus
if total_bad > 0:
    error_summary['Protsent (%)'] = (error_summary['Vigade koguarv'] / total_bad * 100).round(1)
else:
    error_summary['Protsent (%)'] = 0.0

error_summary

,Vahesamm,Vigade koguarv,Protsent (%)
0,Metaandmete filtreerimine,0,0.0
1,RAGi vektorotsing,7,87.5
2,LLMi vastuse genereerimine,1,12.5
